# Sqlite Quickstart Guide

This guide will walk you through the basics of using the Ormophine ORM. We'll create a simple database, define a table, perform CRUD operations, and then dive into a more complex example using `ColumnsOperation` for advanced queries.

> **Note:**
> The ORM uses a thread‑safe, queue‑based architecture. All database access is done through a `Driver` instance; you never interact with a raw `sqlite3.Connection` directly.

## Getting Started

Import the necessary classes:

In [ ]:
from Ormophine.Sqlite import Driver, TableStructure, DataTypes

Connect to a database (creates the file if it doesn't exist):

In [ ]:
db = Driver("mydatabase.db")

After connection, the Driver automatically discovers existing tables and makes them available as attributes (e.g., `db.users`). If the database is new, no tables will exist yet.

## Creating Your First Table

Use `TableStructure` to define a schema, then call `db.create_table()`.

In [ ]:
# Define a 'users' table
users_schema = TableStructure("users", strict=True)

users_schema.add_column(
    "id", DataTypes.INTEGER(),
    primary_key=True
)
users_schema.add_column(
    "username", DataTypes.TEXT(max_length=50),
    unique=True, not_null=True
)
users_schema.add_column(
    "age", DataTypes.INTEGER(min_val=0),
    default_value=18
)

users = db.create_table(users_schema)  # returns a Table object

The `Table` object (`users`) provides methods for all operations on that table.

## Inserting Data

Single insert:

In [ ]:
users.insert({
    users.username: "alice",
    users.age: 25
})
# id is auto‑incremented by SQLite if it's INTEGER PRIMARY KEY

Bulk insert:

In [ ]:
users.bulk_insert(
    columns=[users.username, users.age],
    data_list=[
        ("bob", 30),
        ("charlie", 22),
        ("diana", 28),
    ]
)

## Querying Data

Fetch rows using `get_row()`. You can select specific columns, apply a `WHERE` clause, and order the results.

In [ ]:
# Fetch all usernames and ages, ordered by age
all_users = users.get_row(
    which_columns=[users.username, users.age],
    order_by=users.age
)
for row in all_users:
    print(row)  # ('alice', 25), ('charlie', 22), ...

Using a `WHERE` condition with `ColumnsOperation`:

In [ ]:
# Get users older than 24
condition = users.age > 24
older_users = users.get_row(
    which_columns=[users.username],
    where=condition
)
for username in older_users:
    print(username)  # alice, bob, diana

## Updating Data

Update rows matching a condition:

In [ ]:
# Increase age by 1 for all users under 30
condition = users.age < 30
users.update(
    update={users.age: users.age + 1},  # ColumnsOperation!
    where=condition
)

> **Note:**
> In `update`, you can assign a new value directly (e.g., `{users.age: 30}`) or use a `ColumnsOperation` to refer to existing column values (e.g., `users.age + 1`).

## Deleting Data

Delete rows that match a condition:

In [ ]:
condition = users.username == "charlie"
users.delete_row(where=condition)

## Complex Example with ColumnsOperation

Now let’s work with a `products` table and perform arithmetic and string operations in queries.

1. Create the `products` table:

In [ ]:
products_schema = TableStructure("products", strict=True)
products_schema.add_column("id", DataTypes.INTEGER(), primary_key=True)
products_schema.add_column("name", DataTypes.TEXT())
products_schema.add_column("price", DataTypes.REAL(min_val=0.0))
products_schema.add_column("discount", DataTypes.REAL(min_val=0.0, max_val=1.0))
products_schema.add_column("category", DataTypes.TEXT())

products = db.create_table(products_schema)

2. Insert sample data:

In [ ]:
products.bulk_insert(
    columns=[products.name, products.price, products.discount, products.category],
    data_list=[
        ("Widget", 19.99, 0.1, "gadgets"),
        ("Gadget Pro", 49.99, 0.2, "gadgets"),
        ("SuperTool", 29.99, 0.0, "tools"),
        ("MegaWidget", 99.99, 0.25, "gadgets"),
    ]
)

3. **Arithmetic**: Calculate the final price after discount and select only those with a final price above 30.

In [ ]:
final_price = products.price * (1 - products.discount)  # ColumnsOperation
condition = final_price > 30

# Select product name and computed final_price
result = products.get_row(
    which_columns=[products.name, final_price],
    where=condition,
    order_by=final_price
)

for name, price in result:
    print(f"{name}: ${price:.2f}")
# Output:
# Gadget Pro: $39.99
# MegaWidget: $74.99

4. **String operations**: Find all gadget products whose name contains "Widget", case‑insensitive.

In [ ]:
condition = (products.category == "gadgets") & (products.name.lower().contains("widget"))

gadget_widgets = products.get_row(
    which_columns=[products.name, products.price],
    where=condition
)

for name, price in gadget_widgets:
    print(f"{name}: ${price}")
# Widget: $19.99
# MegaWidget: $99.99

5. **Update using calculation**: Apply an extra 5% discount to all products whose current discount is less than 20%.

In [ ]:
new_discount = products.discount + 0.05  # increase by 5 percentage points
products.update(
    update={products.discount: new_discount},
    where=products.discount < 0.2
)

6. **Deleting with string condition**: Remove all products from the "tools" category.

In [ ]:
products.delete_row(where=products.category == "tools")

## Batch Operations

For multiple dependent statements, use a `batch()` context to run them atomically.

In [ ]:
batch = products.batch()
batch.insert({products.name: "Hammer", products.price: 12.50, products.discount: 0.0, products.category: "tools"})
batch.update(
    update={products.price: products.price * 1.1},  # increase price by 10%
    where=products.discount == 0.0
)
batch.run()  # both statements executed in a single transaction

## Joining Tables

Suppose we have a second table `orders` that references `products.id`.

In [ ]:
orders_schema = TableStructure("orders")
orders_schema.add_column("id", DataTypes.INTEGER(), primary_key=True)
orders_schema.add_column("product_id", DataTypes.INTEGER())
orders_schema.add_column("quantity", DataTypes.INTEGER(min_val=1))
orders = db.create_table(orders_schema)

# Insert some orders
orders.insert({orders.product_id: 1, orders.quantity: 3})
orders.insert({orders.product_id: 2, orders.quantity: 1})
orders.insert({orders.product_id: 4, orders.quantity: 5})

Now join `orders` with `products` to get order details including product name and total cost.

In [ ]:
from Ormophine.Sqlite import Join

# Define join condition
join_condition = orders.product_id == products.id

# Columns to retrieve
columns = [
    orders.id,
    products.name,
    orders.quantity,
    products.price * orders.quantity  # total cost
]

result = orders.join(
    columns=columns,
    joins_list=[Join.Inner(products, join_condition)],
    order_by=orders.id
)

for row in result:
    print(row)  # (order_id, product_name, quantity, total_cost)

This gives you a quick tour of the ORM's capabilities. For a full reference of all methods and options, see the API documentation.